# MERA-XQUAD — Baselines (§3)
M1 zero-shot + M3 translate-test → predictions → `results/`. Cần **Internet** (T_E/NLLB), **GPU** cho nhanh.
Cần Add Input: `hotel-mt5-asqp`, và **gold labeled** (thư mục `hamos26` có `test.json`).


In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# Settings -> bật GPU + Internet. Code lấy trực tiếp từ GitHub (như các notebook khác).
!git clone -q https://github.com/hotuyen21pt/MultiLABSA.git
!pip install -q -r MultiLABSA/requirements-kaggle.txt scipy


In [ ]:
import os, fnmatch
def find_dir_by_name(names, root='/kaggle', required_files=None):
    fallback = None
    for dp, _d, files in os.walk(root, followlinks=True):
        if os.path.basename(dp.rstrip('/\\')) in names:
            if not required_files or any(f in files for f in required_files):
                return dp
            fallback = fallback or dp
    return fallback
def find_file(patterns, root='/kaggle'):
    for dp, _d, files in os.walk(root, followlinks=True):
        for fn in files:
            if any(fnmatch.fnmatch(fn, p) for p in patterns):
                return os.path.join(dp, fn)
    return None

GEN_MODEL_DIR   = find_dir_by_name({'hotel-mt5-asqp','hotel_mt5_asqp'}, required_files=['model.safetensors'])
EXTRACTIVE_CKPT = find_file(['extractive_teacher.pt'])
UNLABELED_CSV   = find_file(['hotel_review_merged.csv','hotel_review*_lang.csv'])
# gold labeled: tìm thư mục chứa test.json (dataset hoặc trong repo nếu data_final được commit)
LABELED_DIR = find_dir_by_name({'hamos26'}, required_files=['test.json']) \
              or '/kaggle/working/MultiLABSA/data_final/labeled_data/hamos26'
RESULTS = '/kaggle/working/results'; os.makedirs(RESULTS, exist_ok=True)
print('GEN_MODEL_DIR  :', GEN_MODEL_DIR)
print('EXTRACTIVE_CKPT:', EXTRACTIVE_CKPT)
print('UNLABELED_CSV  :', UNLABELED_CSV)
print('LABELED_DIR    :', LABELED_DIR)
assert GEN_MODEL_DIR, 'Thiếu hotel-mt5-asqp (Add Input dataset).'
assert os.path.exists(os.path.join(LABELED_DIR,'test.json')), 'Thiếu gold test.json (Add Input labeled data hoặc commit data_final).'


In [ ]:
%cd /kaggle/working/MultiLABSA


## M1 — Zero-shot


In [ ]:
!python -m baselines.run_baseline --baseline zero_shot \
  --input "{LABELED_DIR}/test.json" --generative_model "{GEN_MODEL_DIR}" \
  --out /kaggle/working/preds_m1.json


In [ ]:
for seed in (42, 43, 44):
    !python -m evaluation.run_eval --predictions /kaggle/working/preds_m1.json \
      --labeled_dir "{LABELED_DIR}" --split test --method M1_zeroshot --track zero_shot \
      --seed {seed} --results_dir {RESULTS}


## M3 — Translate-test (dịch sang EN rồi dự đoán; cần Internet tải NLLB)


In [ ]:
!python -m baselines.run_baseline --baseline translate_test \
  --input "{LABELED_DIR}/test.json" --generative_model "{GEN_MODEL_DIR}" \
  --out /kaggle/working/preds_m3.json
!python -m evaluation.run_eval --predictions /kaggle/working/preds_m3.json \
  --labeled_dir "{LABELED_DIR}" --split test --method M3_translate_test \
  --track native_vs_translated --seed 42 --results_dir {RESULTS}


→ `results/` tích luỹ. Chạy notebook **evaluation** để dựng bảng so sánh.
